### Residue Interaction Network Analysis of Proteins


[A must-read review paper]()

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil
import tempfile

DATA_REPOSITORY = "SAMPEL-Group/RIN-framework"
DATA_COMMIT = "eb7397b1aca8c9b190aad92322f4e0aec3a5048d"
DATA_URL = f"https://raw.githubusercontent.com/{DATA_REPOSITORY}/{DATA_COMMIT}/tutorial"
DATA_CACHE = Path.cwd() / ".rin_tutorial_data" / DATA_COMMIT[:12]


def tutorial_data(relative):
    """Reuse an existing file, or download it from GitHub."""
    if Path(relative).is_absolute() or ".." in Path(relative).parts:
        raise ValueError("Use a path relative to the tutorial folder")
    for root in [Path.cwd(), Path.cwd() / "tutorial", Path.cwd().parent / "tutorial", DATA_CACHE]:
        path = root / relative
        if path.is_file():
            return path
    target = DATA_CACHE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        print(f"Downloading {relative}…")
        with urlopen(f"{DATA_URL}/{relative}", timeout=45) as response:
            with tempfile.NamedTemporaryFile(dir=target.parent, delete=False) as output:
                temporary = Path(output.name)
                shutil.copyfileobj(response, output)
        temporary.replace(target)
        return target
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


import numpy as np
import networkx as nx
import MDAnalysis as md
from scipy.spatial import cKDTree


def heavy_atom_rin(protein, cutoff=6.0, sequence_exclusion=3, subunit_id=0):
    """One VP/protein: minimum heavy-atom distance < cutoff, no PBC."""
    if not np.isfinite(cutoff) or cutoff <= 0:
        raise ValueError("cutoff must be a positive distance in angstrom")
    if not isinstance(sequence_exclusion, int) or sequence_exclusion < 0:
        raise ValueError("sequence_exclusion must be a nonnegative integer")
    residues = protein.residues
    if len(set(residues.resids)) != len(residues):
        raise ValueError("Select one VP/protein before constructing this graph")
    heavy = protein.select_atoms("not name H*")
    xyz = heavy.positions.astype(np.float64)
    if not np.isfinite(xyz).all():
        raise ValueError("Nonfinite heavy-atom coordinates")
    atom_to_node = np.searchsorted(residues.ix, heavy.resindices)
    if len(np.unique(atom_to_node)) != len(residues):
        raise ValueError("Every residue must contain a heavy atom")
    atom_pairs = cKDTree(xyz).query_pairs(cutoff, output_type="ndarray")
    # The spatial query includes the boundary; the tutorial uses strict <.
    delta = xyz[atom_pairs[:, 0]] - xyz[atom_pairs[:, 1]]
    atom_pairs = atom_pairs[np.einsum("ij,ij->i", delta, delta) < cutoff**2]
    residue_pairs = np.sort(atom_to_node[atom_pairs], axis=1)
    residue_pairs = residue_pairs[
        residue_pairs[:, 1] - residue_pairs[:, 0] > sequence_exclusion
    ]
    residue_pairs = np.unique(residue_pairs, axis=0)
    graph = nx.Graph()
    graph.add_nodes_from(
        (i, {"subunit_id": subunit_id, "residue_id": int(r.resid), "residue_name": r.resname})
        for i, r in enumerate(residues)
    )
    graph.add_edges_from(residue_pairs.tolist())
    return graph


In [ ]:
# Import Modules
import os
import numpy as np
from numpy import savetxt
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.pylab as pl
from mpl_toolkits.mplot3d import Axes3D
import MDAnalysis as md
from MDAnalysis.analysis.distances import distance_array, contact_matrix
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA
import ipywidgets as widgets
import py3Dmol
from tqdm import tqdm
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
prot = "lysozyme"  # lysozyme, thaumatin, apoferritin
rThresh = 6.0  # heavy-atom cutoff, angstrom

gro = str(tutorial_data(f"conf/{prot}.gro"))
traj = str(tutorial_data(f"conf/{prot}-unfolding.xtc"))
u = md.Universe(gro, traj, refresh_offsets=True)

In [ ]:
import io

heavyProtein = u.select_atoms("protein and not name H*")
with io.StringIO() as pdb_buffer:
    with md.Writer(pdb_buffer, format="PDB", multiframe=True) as writer:
        for ts in u.trajectory:
            writer.write(heavyProtein)
        trajectory_pdb = pdb_buffer.getvalue()

view = py3Dmol.view(width=600, height=400)
view.addModelsAsFrames(trajectory_pdb, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()

def set_frame(frame):
    view.setFrame(frame)
    view.show()

widgets.interact(set_frame, frame=(0, u.trajectory.n_frames - 1))

In [ ]:
# Extract positions
frame = 0 # Set the frame you want to analyze; 0 --> unfolded, 11 --> folded
u.trajectory[frame]

# Identify protein heavy atoms
heavyProtein = u.select_atoms("protein and not name H*")
CA = u.select_atoms("protein and name CA").positions

# Store positions
heavyPos = heavyProtein.positions

# Split into atoms and residues for indexing
heavyAtoms = heavyProtein.split("atom")
heavyResidues = heavyProtein.split("residue")
heavyResids = heavyProtein.resids
n = len(heavyResidues)

### Creating a Graph Representation of a Residue Interaction Network
To construct a graph, we identify all residue-residue contacts that satisfy the following criteria:


1.   Residues *i,j* are greater than 3 residues apart in sequence space.
2.   Residues *i,j* are within a cut-off distance defined by `rThresh`.

Each residue is considered a node, with a position (irrelevant) defined by its alpha carbon. Edges are drawn between nodes that satisfy the above criteria.



In [ ]:
# Construct the same minimum-heavy-atom contact network using a spatial query.
protein = u.select_atoms("protein")
contact_graph = heavy_atom_rin(protein, cutoff=rThresh, sequence_exclusion=3)
nodes = list(contact_graph.nodes)
uniqueEdges = list(contact_graph.edges)

In [ ]:
# Construct graph
G = nx.Graph() # Initialize graph
G.add_nodes_from(nodes) # Add nodes
G.add_edges_from(uniqueEdges) # Add edges for a given cut-off

In [ ]:
# Visualize graph
import plotly.graph_objects as go

labels = [f"{heavyResidues[k].resnames[0]}{heavyResidues[k].resids[0]}" for k in range(n)]

edge_x, edge_y, edge_z = [], [], []
for i, j in G.edges():
    edge_x += [CA[i,0], CA[j,0], None]
    edge_y += [CA[i,1], CA[j,1], None]
    edge_z += [CA[i,2], CA[j,2], None]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=list(range(n)), colorscale='Rainbow'),
                            text=labels, hoverinfo='text'))
fig.show()

### Centrality Measures

**Degree centrality** is the simplest measure: for each residue $i$, it's the number of contacts (edges) divided by the maximum possible.

$$C_D(i) = \frac{\deg(i)}{n-1}$$

where $\deg(i)$ is the number of edges connected to residue $i$ and $n$ is the total number of residues. Residues with high degree centrality are hubs with many simultaneous contacts — often buried core residues with dense packing.

**Closeness centrality** is the inverse of the average shortest-path distance from residue $i$ to all other residues.

$$C_C(i) = \frac{n-1}{\sum_{j \neq i} d(i,j)}$$

where $d(i,j)$ is the shortest-path distance (number of edges) between residues $i$ and $j$. Residues with high closeness can reach the rest of the structure in few steps, often corresponding to geometrically central positions in the fold.

**Betweenness centrality** quantifies how often residue $i$ lies on the shortest path between other pairs of residues.

$$C_B(i) = \sum_{s \neq i \neq t} \frac{\sigma_{st}(i)}{\sigma_{st}}$$

where $\sigma_{st}$ is the total number of shortest paths between residues $s$ and $t$, and $\sigma_{st}(i)$ is the number of those paths that pass through $i$. Residues with high betweenness often act as bridges between different parts of the network, even if they have relatively few contacts. (low degree).

In [ ]:
# Betweenness centrality
betweenness = nx.betweenness_centrality(G)
bc_vals = [betweenness[k] for k in range(n)]
bc_labels = [f"{labels[k]}<br>BC: {bc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=bc_vals, colorscale='Magma', colorbar=dict(title="Betweenness")),
                            text=bc_labels, hoverinfo='text'))
fig.update_layout(title="Betweenness Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Closeness centrality
closeness = nx.closeness_centrality(G)
cc_vals = [closeness[k] for k in range(n)]
cc_labels = [f"{labels[k]}<br>CC: {cc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=cc_vals, colorscale='Magma', colorbar=dict(title="Closeness")),
                            text=cc_labels, hoverinfo='text'))
fig.update_layout(title="Closeness Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Degree centrality
degree = nx.degree_centrality(G)
dc_vals = [degree[k] for k in range(n)]
dc_labels = [f"{labels[k]}<br>DC: {dc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=dc_vals, colorscale='Magma', colorbar=dict(title="Degree")),
                            text=dc_labels, hoverinfo='text'))
fig.update_layout(title="Degree Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Show centrality measurements correlation
fig, axes = plt.subplots(1, 3, figsize=(12,4))

axes[0].scatter(bc_vals, cc_vals, s=15, c="maroon")
axes[0].set_xlabel("Betweenness")
axes[0].set_ylabel("Closeness")

axes[1].scatter(cc_vals, dc_vals, s=15, c="maroon")
axes[1].set_xlabel("Closeness")
axes[1].set_ylabel("Degree")

axes[2].scatter(dc_vals, bc_vals, s=15, c="maroon")
axes[2].set_xlabel("Degree")
axes[2].set_ylabel("Betweenness")

plt.tight_layout()
plt.show()

In [ ]:
# Plot centrality measurements for all residues
fig, axes = plt.subplots(3, 1, figsize=(10,5), sharex=True)

data = [(bc_vals, "Betweenness"), (cc_vals, "Closeness"), (dc_vals, "Degree")]

for ax, (vals, name) in zip(axes, data):
    ax.plot(nodes, vals, c="black")
    ax.set_ylabel(name)

    top5 = np.argsort(vals)[-5:]
    for idx in top5:
        ax.scatter(idx, vals[idx], marker='s', c="firebrick", zorder=3)
        resname = heavyResidues[idx].resnames[0].capitalize()
        resid = heavyResidues[idx].resids[0]
        ax.annotate(f"{resname}{resid}", (idx, vals[idx]), textcoords="offset points", xytext=(0,8), ha='center', fontsize=8)
    ax.set_xlim(0, n - 1)

axes[2].set_xlabel("Residue index")

plt.tight_layout()
plt.show()

### Embedding Information on Edges

So far, edges have only encoded *whether* a contact exists. In practice, edges can carry additional information — contact strength, frequency over a trajectory, interaction energy, or any other pairwise property — as edge weights. Below, we assign synthetic weights to demonstrate how this changes the graph: weighted betweenness centrality recomputes shortest paths using these weights rather than treating every edge as equal, which can shift which residues appear as critical "bridges" in the network. I and others have used this approach to build RIN variants, such as energetic or dynamic RINs.

In [ ]:
# Reconstruct graph with synthetic edge weights and weighted betweenness centrality
G_weighted = nx.Graph()
G_weighted.add_nodes_from(nodes)

rng = np.random.default_rng(42)
for i, j in uniqueEdges:
    weight = rng.uniform(0.1, 1.0)  # synthetic edge weight (e.g., could represent contact strength, frequency, dynamic cross-correlations, etc.)
    G_weighted.add_edge(i, j, weight=weight)

# Weighted betweenness centrality (NetworkX treats 'weight' as distance by default --
# higher weight = longer path, so invert if weight should represent "strength")
bc_weighted = nx.betweenness_centrality(G_weighted, weight='weight')
bc_w_vals = [bc_weighted[k] for k in range(n)]
bc_w_labels = [f"{labels[k]}<br>BC: {bc_w_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=bc_w_vals, colorscale='Magma', colorbar=dict(title="Weighted Betweenness")),
                            text=bc_w_labels, hoverinfo='text'))
fig.update_layout(title="Betweenness Centrality (Synthetic Edge Weights)")
fig.update_layout(showlegend=False)
fig.show()

## AAV8 capsid

One MD frame (334; 400.8 ns). 

In [ ]:
import io
import json
import py3Dmol
import plotly.graph_objects as go

VP = 0
HEAVY_CUTOFF = 6.0
SEQUENCE_EXCLUSION = 3

aav8_metadata = json.loads(tutorial_data("aav8/metadata.json").read_text())
capsid_universe = md.Universe(
    str(tutorial_data("aav8/aav8_capsid_frame334.gro"))
)
capsid_protein = capsid_universe.select_atoms("protein")

atoms_per_vp = aav8_metadata["single_vp_atoms"]
n_vps = aav8_metadata["full_capsid_subunits"]
assert 0 <= VP < n_vps

protein = capsid_protein[VP * atoms_per_vp:(VP + 1) * atoms_per_vp]
heavyProtein = protein.select_atoms("not name H*")
heavyResidues = heavyProtein.split("residue")
CA = protein.select_atoms("name CA").positions
n = len(heavyResidues)

### Single VP

In [ ]:
with io.StringIO() as pdb_buffer:
    with md.Writer(pdb_buffer, format="PDB", bonds=None) as writer:
        writer.write(heavyProtein)
        vp_pdb = pdb_buffer.getvalue()

view = py3Dmol.view(width=600, height=400)
view.addModel(vp_pdb, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

### Single-VP RIN

Same as lysozyme: minimum heavy-atom distance <6 Å; exclude sequence neighbors ≤3. 

In [ ]:
G = heavy_atom_rin(
    protein,
    cutoff=HEAVY_CUTOFF,
    sequence_exclusion=SEQUENCE_EXCLUSION,
    subunit_id=VP,
)
nodes = list(G.nodes)
uniqueEdges = list(G.edges)

labels = [
    f"{residue.resnames[0]}{residue.resids[0]}"
    for residue in heavyResidues
]

edge_x, edge_y, edge_z = [], [], []
for i, j in G.edges():
    edge_x += [CA[i, 0], CA[j, 0], None]
    edge_y += [CA[i, 1], CA[j, 1], None]
    edge_z += [CA[i, 2], CA[j, 2], None]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode="lines", line=dict(color="gray", width=2),
    hoverinfo="none",
))
fig.add_trace(go.Scatter3d(
    x=CA[:, 0], y=CA[:, 1], z=CA[:, 2],
    mode="markers",
    marker=dict(size=4, color=list(range(n)), colorscale="Rainbow"),
    text=labels, hoverinfo="text",
))
fig.update_layout(title=f"AAV8 · VP{VP} RIN", showlegend=False)
fig.show()

print(f"{G.number_of_nodes()} nodes · {G.number_of_edges()} edges")

In [ ]:
# Betweenness centrality
betweenness = nx.betweenness_centrality(G)
bc_vals = [betweenness[k] for k in range(n)]
bc_labels = [f"{labels[k]}<br>BC: {bc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=bc_vals, colorscale='Magma', colorbar=dict(title="Betweenness")),
                            text=bc_labels, hoverinfo='text'))
fig.update_layout(title="Betweenness Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Closeness centrality
closeness = nx.closeness_centrality(G)
cc_vals = [closeness[k] for k in range(n)]
cc_labels = [f"{labels[k]}<br>CC: {cc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=cc_vals, colorscale='Magma', colorbar=dict(title="Closeness")),
                            text=cc_labels, hoverinfo='text'))
fig.update_layout(title="Closeness Centrality")
fig.update_layout(showlegend=False)
fig

In [ ]:
# Degree centrality
degree = nx.degree_centrality(G)
dc_vals = [degree[k] for k in range(n)]
dc_labels = [f"{labels[k]}<br>DC: {dc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=dc_vals, colorscale='Magma', colorbar=dict(title="Degree")),
                            text=dc_labels, hoverinfo='text'))
fig.update_layout(title="Degree Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Plot centrality measurements and highlight the top 10 residues.
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)

data = [(bc_vals, "Betweenness"), (cc_vals, "Closeness"), (dc_vals, "Degree")]

for ax, (vals, name) in zip(axes, data):
    ax.plot(nodes, vals, c="black")
    ax.set_ylabel(name)

    top10 = np.argsort(vals)[-10:]
    for idx in top10:
        ax.scatter(nodes[idx], vals[idx], marker="s", c="firebrick", zorder=3)
        resname = heavyResidues[idx].resnames[0].capitalize()
        resid = heavyResidues[idx].resids[0]
        ax.annotate(
            f"{resname}{resid}",
            (nodes[idx], vals[idx]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8)

    ax.set_xlim(0, n - 1)

axes[2].set_xlabel("Residue index")
plt.tight_layout()
plt.show()

for vals, name in data:
    print(f"\n{name} — Top 10")
    print(f"{'Rank':<6}{'Residue':<12}{'Value':>12}")

    top10 = np.argsort(vals)[-10:][::-1]

    for rank, idx in enumerate(top10, start=1):
        resname = heavyResidues[idx].resnames[0].capitalize()
        resid = heavyResidues[idx].resids[0]
        label = f"{resname}{resid}"

        print(f"{rank:<6}{label:<12}{vals[idx]:>12.6f}")

### Complete capsid
60 VPs.

In [ ]:
view = py3Dmol.view(width=800, height=600)
palette = ["#4c78a8", "#72b7b2", "#f2cf5b",
           "#f58518", "#b279a2", "#8cbf88"]

for vp_id in range(n_vps):
    atoms = capsid_protein[
        vp_id * atoms_per_vp:(vp_id + 1) * atoms_per_vp
    ]
    backbone = atoms.select_atoms("name N CA C O OT1")

    with io.StringIO() as pdb_buffer:
        with md.Writer(pdb_buffer, format="PDB", bonds=None) as writer:
            writer.write(backbone)
            vp_pdb = pdb_buffer.getvalue()

    view.addModel(vp_pdb, "pdb")
    view.setStyle(
        {"model": vp_id},
        {"cartoon": {"color": palette[vp_id % len(palette)]}},
    )

view.zoomTo()
view.show()

In [ ]:
capsid_ca = capsid_protein.select_atoms("name CA")
capsid_heavy = capsid_protein.select_atoms("not name H*")
capsid_xyz = capsid_ca.positions.astype(np.float64)

node_vp = capsid_ca.indices // aav8_metadata["single_vp_atoms"]
capsid_nodes = np.column_stack((node_vp, capsid_ca.resids))
assert len(capsid_ca) == len(capsid_protein.residues)

xyz = capsid_heavy.positions.astype(np.float64)
atom_node = np.searchsorted(capsid_ca.resindices, capsid_heavy.resindices)
tree = cKDTree(xyz)
n_nodes = len(capsid_nodes)
edge_blocks = []

for start in range(0, len(xyz), 4096):
    neighbors = tree.query_ball_point(xyz[start:start + 4096], HEAVY_CUTOFF)
    atom_i = np.repeat(
        np.arange(start, start + len(neighbors)),
        [len(x) for x in neighbors],
    )
    atom_j = np.concatenate(neighbors).astype(np.int64)
    keep = atom_i < atom_j
    atom_i, atom_j = atom_i[keep], atom_j[keep]

    delta = xyz[atom_i] - xyz[atom_j]
    keep = np.einsum("ij,ij->i", delta, delta) < HEAVY_CUTOFF**2
    i, j = atom_node[atom_i[keep]], atom_node[atom_j[keep]]

    # Exclude sequence neighbors only within the same VP.
    keep = (node_vp[i] != node_vp[j]) | (j - i > SEQUENCE_EXCLUSION)
    edge_blocks.append(np.unique(i[keep] * n_nodes + j[keep]))

edge_keys = np.unique(np.concatenate(edge_blocks))
capsid_edges = np.column_stack((edge_keys // n_nodes, edge_keys % n_nodes))
del tree, edge_blocks, neighbors, atom_i, atom_j, delta

print(f"Nodes: {len(capsid_nodes):,} | Edges: {len(capsid_edges):,}")

In [ ]:
edge_xyz = np.full((len(capsid_edges), 3, 3), np.nan)
edge_xyz[:, :2] = capsid_xyz[capsid_edges]
edge_xyz = edge_xyz.reshape(-1, 3)

capsid_rin_view = go.Figure([
    go.Scatter3d(
        x=edge_xyz[:, 0], y=edge_xyz[:, 1], z=edge_xyz[:, 2],
        mode="lines", line=dict(color="#a8b0ba", width=1),
        opacity=0.3, hoverinfo="skip", showlegend=False),
    go.Scatter3d(
        x=capsid_xyz[:, 0], y=capsid_xyz[:, 1], z=capsid_xyz[:, 2],
        mode="markers",
        marker=dict(size=2, color=node_vp, colorscale="Turbo", showscale=False),
        customdata=capsid_nodes,
        hovertemplate="VP%{customdata[0]} · residue %{customdata[1]}<extra></extra>",
        showlegend=False),
])
capsid_rin_view.update_layout(
    title="AAV8 · full-capsid RIN", height=620,
    margin=dict(l=0, r=0, t=40, b=0),
    scene=dict(
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        zaxis=dict(visible=False), aspectmode="data"),paper_bgcolor="white",)
capsid_rin_view.show(config=dict(displaylogo=False, scrollZoom=True))